# Public-source note

This notebook preserves the original 22-state histone ChromHMM reordering logic. Notebook outputs were stripped and private server paths were replaced with `/path/to/...` placeholders. Scientific state order and labels were not changed.

In [ ]:
import os, shlex, subprocess, sys
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap


In [ ]:
learnmodel_out_dir = "/path/to/CEBPB_seq/ChromHMM/All_cellTypes_histones/numstates_22/"
reordered_out_dir = "/path/to/CEBPB_seq/ChromHMM/All_cellTypes_histones/numstates_22_reordered/"

emissions_in = Path(learnmodel_out_dir) / "emissions_22.txt"
emissions_out = Path(reordered_out_dir) / "emissions_22.txt"


seg_in = Path(learnmodel_out_dir) / "Histones_22_segments.bed"
seg_out= Path(reordered_out_dir) / "Histones_22_segments.bed"

emissions_out_pdf= Path(reordered_out_dir) / "Histones_STACKED_ChromHMM_22_reordered.pdf"

In [ ]:
emissions_out

In [ ]:

marks=[
'H3K9ac', 
'H3K4me3', 
'H3K4me2', 
'H3K4me1', 
'H3K27ac',
'H3K27me3', 
'H3K79me2',
'H3K36me3', 
'H3K9me3']


cell_types=["MEF",
          "NSC",
            "ESC",
            "BMpreB"
           ]
col_order=[f"{cell}_{mark}"  for mark in marks for cell in cell_types]
col_order

In [ ]:
desired_order=[7,6,16,
               2,4,14,15,1,3,5,8,
  9, 10, 11, 12, 13,   22, 
               17,  20, 
               18,
               21, 19]
state_map = {old: new for new, old in enumerate(desired_order, start=1)}
state_map

In [ ]:
# ---- create directory if it does not exist ----
reordered_path = Path(reordered_out_dir)

reordered_path.mkdir(parents=True, exist_ok=True)

print(f"Directory ready: {reordered_path}")

In [ ]:
# emissions_df=pd.read_csv(emissions_in, sep="\t")

# state_col = "State (Emission order)"
# emissions_df[state_col] = emissions_df[state_col].astype(int)
# emissions_df = emissions_df.set_index(state_col)
# emissions_df["new_state"] = emissions_df.index.map(state_map)
# emissions_df = emissions_df.sort_values("new_state")

# emissions_df=emissions_df[col_order]
# em_mat = emissions_df[col_order]


In [ ]:
import pandas as pd

# load
emissions_df = pd.read_csv(emissions_in, sep="\t")

state_col = "State (Emission order)"

# ensure numeric
emissions_df[state_col] = emissions_df[state_col].astype(int)

# set old state number as index
emissions_df = emissions_df.set_index(state_col)

# map old state number → new state number
emissions_df["new_state"] = emissions_df.index.map(state_map)

# sort by new state
emissions_df = emissions_df.sort_values("new_state")

# ---- IMPORTANT PART ----
# replace index with new state numbers
emissions_df.index = emissions_df["new_state"]

# drop helper column
emissions_df = emissions_df.drop(columns=["new_state"])

# optional: rename index to ChromHMM style labels
emissions_df.index = [f"H{i}" for i in emissions_df.index]

# reorder columns
emissions_df = emissions_df[col_order]

# ---- save ----
emissions_df.to_csv(emissions_out, sep="\t")

print("emissions.txt reordered and saved.")

# matrix for plotting
em_mat = emissions_df.copy()
em_mat


In [ ]:
# Keep only numeric columns (the marks)
em_mat = em_mat.select_dtypes(include=[np.number]).copy()

# ChromHMM white to blue colormap
chromhmm_cmap = LinearSegmentedColormap.from_list(
    "chromhmm_white_blue",
    ["#FFFFFF", "#DEEBF7", "#9ECAE1", "#4292C6", "#08519C"]
)

# plt.figure(figsize=(1.1 * em_mat.shape[1] + 4, 0.35 * em_mat.shape[0] + 4))
# plt.figure(figsize=(20,5))

    
plt.figure(
    figsize=(0.3 * em_mat.shape[1] + 4,
             0.2 * em_mat.shape[0] + 4)
)

ax = sns.heatmap(
    em_mat,
    cmap=chromhmm_cmap,
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.3,
    cbar=False,
    linecolor="lightgray",
    # cbar_kws={"label": "Emission probability"}
)

ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_xlabel("Mark")
ax.set_ylabel("State")
ax.set_title("ChromHMM Emissions")

plt.tight_layout()
# plt.savefig("chromhmm_emissions_heatmap.pdf")
# plt.savefig("chromhmm_emissions_heatmap.png", dpi=300)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

def plot_emissions_with_rowgroups(em_mat, row_groups=None):
    chromhmm_cmap = LinearSegmentedColormap.from_list(
        "chromhmm_white_blue",
        ["#FFFFFF", "#DEEBF7", "#9ECAE1", "#4292C6", "#08519C"]
    )

    mark_labels = [mark for mark in marks for _ in cell_types]
    cell_labels = cell_types * len(marks)

    # Reorder rows by row_groups (keys are group names, values are row ids)
    # Assumes your values match em_mat.index labels (example: states labeled 1..K)
    # If your em_mat index is 0..K-1, then pass values as 0 based, or adjust here.
    group_boundaries = []
    group_centers = []

    if row_groups is not None:
        ordered_rows = []
        group_sizes = []

        for gname, rows in row_groups.items():
            ordered_rows.extend(rows)
            group_sizes.append((gname, len(rows)))

        em_mat = em_mat.loc[ordered_rows]

        # compute boundaries and centers in heatmap coordinates
        running = 0
        for gname, sz in group_sizes:
            start = running
            end = running + sz
            group_boundaries.append(end)  # draw line at end of group
            group_centers.append((gname, (start + end) / 2.0))
            running = end

    fig =plt.figure(
        figsize=(0.5 * em_mat.shape[1] + 4,
                 0.35 * em_mat.shape[0] + 4)
    )

    ax = sns.heatmap(
        em_mat,
        cmap=chromhmm_cmap,
        vmin=0,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.3,
        cbar=False,
        linecolor="lightgray",
        cbar_kws={"label": "Emission probability"}
    )

    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    # Bottom labels: cell types
    ax.set_xticks(np.arange(len(col_order)) + 0.5)
    ax.set_xticklabels(cell_labels, rotation=90, fontsize=20)

    # Top labels: marks (centered)
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())

    mark_centers = []
    for i in range(len(marks)):
        start = i * len(cell_types)
        end = start + len(cell_types)
        mark_centers.append((start + end) / 2.0)

    ax_top.set_xticks(mark_centers)
    ax_top.set_xticklabels(marks, fontsize=20)
    ax_top.tick_params(axis="x", length=0)

    ax.set_xlabel("")
    ax_top.set_xlabel("")

    ax.tick_params(axis="y", labelsize=20)

    # Vertical separators between marks
    for i in range(1, len(marks)):
        ax.axvline(i * len(cell_types), color="black", linewidth=1)

    # Row group separators and row group labels
    if row_groups is not None:
        # horizontal separators
        for b in group_boundaries[:-1]:
            ax.axhline(b, color="black", linewidth=1)

        # group labels on left margin, keep state tick labels as is
        # x is in axes coords, y is in data coords
        for gname, ycenter in group_centers:
            ax.text(
                x=-0.05, y=ycenter,
                s=gname,
                transform=ax.get_yaxis_transform(),
                ha="right", va="center",
                fontsize=20,
                fontweight="bold"
            )

        # add a bit more left margin so group labels are visible
        plt.subplots_adjust(left=0.22)

    plt.tight_layout()
    return fig
    # plt.show()


In [ ]:
# desired_order=[7,6,16,
#                2,4,14,15,1,3,5,8,
#   9, 10, 11, 12, 13,   22, 
#                17,  20,
#                18,
#                 21, 
#                19]

In [ ]:
emissions_out_pdf

In [ ]:
row_groups = {"Prom":['H1','H2','H3'],
"MEF Enh":[ 'H4'],
"NSC Enh":[ 'H5'],
  "ESC Enh":['H6','H7'],
  "Bcell Enh":[ 'H8'],
"MEF NSC Enh":[ 'H9'],
"MEF NSC Bcell Enh":[ 'H10','H11'],

"TX":[  'H12', 'H13', 'H14', 'H15', 'H16',   'H17'],
"Rep":['H18', 'H19'],
  "TX Rep":['H20'],
"Low Sig":['H21','H22']}

fig =plot_emissions_with_rowgroups(em_mat, row_groups=row_groups)
fig.savefig(emissions_out_pdf, bbox_inches="tight")


In [ ]:
emissions_out

In [ ]:
seg_in

In [ ]:
seg = pd.read_csv(
    seg_in,
    sep="\t",
    header=None,
    usecols=[0,1,2,3],
    names=["chr","start","end","state"]
)

In [ ]:
seg.head()

In [ ]:
import re

def extract_state_number(x):
    x = str(x)
    if x.startswith("E"):
        return int(x[1:])
    return int(x)

seg["state_num"] = seg["state"].apply(extract_state_number)
seg["new_state_num"] = seg["state_num"].map(state_map)
seg["state"] = "H" + seg["new_state_num"].astype(int).astype(str)


In [ ]:
seg.head()

In [ ]:
seg[["chr","start","end","state"]].to_csv(
    seg_out,
    sep="\t",
    header=False,
    index=False
)

print("Segmentation BED reordered and saved.")


In [ ]:
seg[seg['state']=='H13'].head()